# ЛР № 1. Смешение углей для ТЭС

Вариант 14

Нужно закупить 1000 т угля у пяти поставщиков. Ищу самый дешёвый план, который укладывается в нормы по зольности и сере.

**Среда выполнения → Выполнить всё** запускает весь расчёт.

In [1]:
%pip install -q numpy==2.3.5 scipy==1.17.0 matplotlib==3.10.8

## Данные

Цены, лимиты, зольность и содержание серы получены небольшим случайным разбросом от шаблона.

Читаю `data.json`. Если файла нет, создаю его с `seed=10011`.

In [2]:
import csv
import json
import os
from contextlib import redirect_stderr, redirect_stdout
from ctypes import CDLL
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from scipy.optimize import linprog

SEED = 10011
TOL = 1e-7
STATUS = {0: "optimal", 1: "limit", 2: "infeasible", 3: "unbounded", 4: "solver_error"}

LAB_DIR = Path.cwd()
if LAB_DIR.name != "lr1":
    LAB_DIR = LAB_DIR / "lr1"
DATA_PATH = LAB_DIR / "data.json"
RESULTS = LAB_DIR / "results"
STAR_RESULTS = RESULTS / "star"


def coal(rng):
    return {
        "seed": SEED,
        "suppliers": ["S1", "S2", "S3", "S4", "S5"],
        "price": ([5200, 5400, 5850, 6350, 6500] + rng.integers(-35, 36, 5)).tolist(),
        "capacity": ([380, 370, 320, 300, 240] + rng.integers(-9, 10, 5)).tolist(),
        "ash_pct": ((np.array([170, 80, 104, 53, 131]) + rng.integers(-2, 3, 5)) / 10).tolist(),
        "sulfur_pct": ((np.array([65, 175, 93, 69, 34]) + rng.integers(-2, 3, 5)) / 100).tolist(),
        "demand": 1000,
        "ash_limit_pct": 11.0,
        "sulfur_limit_pct": 1.04,
        "stress_ash_limit_pct": 7.9,
    }


if not DATA_PATH.is_file():
    LAB_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(np.random.SeedSequence([SEED, 1]))
    DATA_PATH.write_text(
        json.dumps(coal(rng), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
data = json.loads(DATA_PATH.read_text(encoding="utf-8"))

## Решатель и файлы

HiGHS решает модель. Таблицы и лог записываются в `results`.

In [3]:
def write_csv(folder, name, headers, rows):
    path = folder / f"{name}.csv"
    path.parent.mkdir(exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(headers)
        writer.writerows(rows)


def write_check(folder, name, result):
    path = folder / f"{name}.json"
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def solve_lp(cost, *, label, verbose=True, **kwargs):
    print(f"\n=== {label} ===", flush=True)
    result = linprog(
        cost, method="highs-ds", options={"presolve": True, "disp": verbose}, **kwargs
    )
    print(f"{STATUS.get(result.status, result.status)} | {result.message}", flush=True)
    return result


def require_status(result, *allowed):
    if result.status not in allowed:
        raise RuntimeError(result.message)


def run_with_log(run, data, folder):
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / "solver.log").open("w", encoding="utf-8", buffering=1) as log:
        # HiGHS пишет лог напрямую в stdout, минуя Python.
        stdout_fd = os.dup(1)
        try:
            os.dup2(log.fileno(), 1)
            with redirect_stdout(log), redirect_stderr(log):
                return run(data)
        finally:
            CDLL(None).fflush(None)
            os.dup2(stdout_fd, 1)
            os.close(stdout_fd)


def show_table(path, columns, labels=None, *, scenario=None):
    names = {"optimal": "Оптимум", "infeasible": "Нет допустимого плана",
             "feasible": "Допустим", **(labels or {})}
    with path.open(encoding="utf-8", newline="") as file:
        rows = [[row[key] for key in columns] for row in csv.DictReader(file)
                if scenario is None or row["scenario"] == scenario]

    def format_cell(value):
        if not value:
            return "—"
        try:
            number = float(value)
        except ValueError:
            return names.get(value, value)
        return f"{number:,.3f}".rstrip("0").rstrip(".").replace(",", " ").replace(".", ",")

    table = [list(columns.values()), ["---"] * len(columns)]
    table += [[format_cell(value) for value in row] for row in rows]
    display(Markdown("\n".join("| " + " | ".join(row) + " |" for row in table)))

## Модель

На каждого поставщика приходится одна переменная. Сумма закупок равна спросу. Зольность и серу смеси считаю как средние, взвешенные по массе.

In [4]:
def inputs(data):
    return tuple(np.array(data[key], dtype=float) for key in
                 ("price", "capacity", "ash_pct", "sulfur_pct"))


def build_model(capacity, ash, sulfur, demand, ash_limit, sulfur_limit):
    return dict(A_ub=np.vstack((ash, sulfur)), b_ub=demand * np.array([ash_limit, sulfur_limit]),
                A_eq=np.ones((1, len(capacity))), b_eq=[demand],
                bounds=np.column_stack((np.zeros(len(capacity)), capacity)))

## Проверка плана

В NumPy отдельно проверяю ограничения и пересчитываю стоимость. Допуск — $10^{-7}$.

In [5]:
def max_violation(values):
    return float(np.max(values, initial=0))


def verify_objective(checks, expected, actual):
    checks["objective_recomputed"] = float(actual)
    checks["objective_error"] = float(abs(expected - actual))
    if any(value > TOL for key, value in checks.items() if key != "objective_recomputed"):
        raise ValueError(f"Independent check failed: {checks}")
    return checks


def verify(plan, price, capacity, ash, sulfur, demand, ash_limit, sulfur_limit, objective):
    checks = {"volume": abs(float(plan.sum()) - demand),
              "upper": max_violation(plan - capacity), "lower": max_violation(-plan),
              "ash_pct": max(float(ash @ plan / demand - ash_limit), 0),
              "sulfur_pct": max(float(sulfur @ plan / demand - sulfur_limit), 0)}
    return verify_objective(checks, objective, price @ plan)


def fill_cheapest(weight, capacity, demand):
    # Один баланс и верхние границы: заполняем объём в порядке роста коэффициента.
    plan = np.zeros_like(capacity)
    remaining = demand
    for i in np.argsort(weight):
        plan[i] = min(capacity[i], remaining)
        remaining -= plan[i]
    if remaining > TOL:
        raise ValueError("Поставщики не покрывают заданный объём")
    return plan


def deficit_capacity(capacity):
    reduced = capacity.copy()
    reduced[1] *= 0.8
    return reduced

## Что меняю в сценариях

В дефиците снижаю лимит S2 на 20%. В стрессе дополнительно ужесточаю норму по зольности. Теневые цены сверяю отдельными небольшими изменениями.

In [6]:
def run_lab(data):
    price, capacity, ash, sulfur = inputs(data)
    demand, ash_limit, sulfur_limit = (data[key] for key in
                                      ("demand", "ash_limit_pct", "sulfur_limit_pct"))
    reduced = deficit_capacity(capacity)
    increased = capacity.copy()
    increased[0] += 1
    scenarios = [
        ("base", capacity, ash_limit, sulfur_limit, 0),
        ("deficit", reduced, ash_limit, sulfur_limit, 0),
        ("stress", reduced, data["stress_ash_limit_pct"], sulfur_limit, 2),
        ("s1_plus_one", increased, ash_limit, sulfur_limit, 0),
        ("ash_plus", capacity, ash_limit + 0.1, sulfur_limit, 0),
        ("sulfur_plus", capacity, ash_limit, sulfur_limit + 0.01, 0),
    ]
    checks, summary, plans, marginals = {}, [], [], []
    for name, cap, a_max, s_max, expected in scenarios:
        result = solve_lp(price, label=name, **build_model(cap, ash, sulfur, demand, a_max, s_max))
        require_status(result, expected)
        if result.status == 2:
            lower_bound = float(ash @ fill_cheapest(ash, cap, demand) / demand)
            assert lower_bound > a_max + TOL
            checks[name] = {"min_ash_without_sulfur": lower_bound, "ash_limit": a_max}
            summary.append([name, cap[1], a_max, s_max, STATUS[result.status], "", "", "", result.nit])
            continue
        plan = result.x
        checks[name] = verify(plan, price, cap, ash, sulfur, demand, a_max, s_max, result.fun)
        summary.append([name, cap[1], a_max, s_max, STATUS[result.status], result.fun,
                        ash @ plan / demand, sulfur @ plan / demand, result.nit])
        plans.extend([name, supplier, cap[i], plan[i], plan[i] * price[i]]
                     for i, supplier in enumerate(data["suppliers"]))
        if name == "base":
            for k, quality in enumerate(("ash_limit_pct", "sulfur_limit_pct")):
                marginals.append([quality, result.ineqlin.residual[k] / demand,
                                  result.ineqlin.marginals[k] * demand])
            marginals.append(["volume_at_fixed_quality", 0, result.eqlin.marginals[0]
                              + result.ineqlin.marginals @ [ash_limit, sulfur_limit]])
            for i, supplier in enumerate(data["suppliers"]):
                marginals.append([supplier, result.upper.residual[i], result.upper.marginals[i]])
                marginals.append([f"{supplier}_lower", result.lower.residual[i], result.lower.marginals[i]])
    write_csv(RESULTS, "scenarios",
              ["scenario", "s2_capacity", "ash_limit", "sulfur_limit", "status", "cost",
               "actual_ash", "actual_sulfur", "iterations"], summary)
    write_csv(RESULTS, "solution", ["scenario", "supplier", "capacity", "tonnes", "cost"], plans)
    write_csv(RESULTS, "marginals", ["constraint", "slack", "marginal"], marginals)
    write_check(RESULTS, "checks", checks)


run_with_log(run_lab, data, RESULTS)

## Что получилось

В первой таблице собраны сценарии, во второй — закупка по поставщикам для базового плана. Полные планы, теневые цены и проверка сохранены в `results`.

In [7]:
show_table(
    RESULTS / "scenarios.csv",
    {"scenario": "Сценарий", "s2_capacity": "Лимит S2, т", "ash_limit": "Зольность, %",
     "sulfur_limit": "Сера, %", "status": "Статус", "cost": "Стоимость, руб."},
    {"base": "Базовый", "deficit": "Дефицит", "stress": "Стресс", "s1_plus_one": "Лимит S1 + 1 т",
     "ash_plus": "Предел зольности + 0,1 п. п.", "sulfur_plus": "Предел серы + 0,01 п. п."},
)
show_table(
    RESULTS / "solution.csv",
    {"supplier": "Поставщик", "tonnes": "Базовая закупка, т", "cost": "Стоимость, руб."},
    scenario="base",
)

checks = json.loads((RESULTS / "checks.json").read_text(encoding="utf-8"))
error = max(item["objective_error"] for item in checks.values() if "objective_error" in item)
print(f"Максимальное расхождение при пересчёте стоимости: {error:.2g}.")

| Сценарий | Лимит S2, т | Зольность, % | Сера, % | Статус | Стоимость, руб. |
| --- | --- | --- | --- | --- | --- |
| Базовый | 376 | 11 | 1,04 | Оптимум | 5 593 927,189 |
| Дефицит | 300,8 | 11 | 1,04 | Оптимум | 5 613 758,805 |
| Стресс | 300,8 | 7,9 | 1,04 | Нет допустимого плана | — |
| Лимит S1 + 1 т | 376 | 11 | 1,04 | Оптимум | 5 593 548,151 |
| Предел зольности + 0,1 п. п. | 376 | 11,1 | 1,04 | Оптимум | 5 587 230,37 |
| Предел серы + 0,01 п. п. | 376 | 11 | 1,05 | Оптимум | 5 586 665,388 |

| Поставщик | Базовая закупка, т | Стоимость, руб. |
| --- | --- | --- |
| S1 | 385 | 2 010 085 |
| S2 | 336,04 | 1 825 703,784 |
| S3 | 40,771 | 237 779,214 |
| S4 | 238,189 | 1 520 359,191 |
| S5 | 0 | 0 |

Максимальное расхождение при пересчёте стоимости: 9.3e-10.


## Со звёздочкой. Эластичный фильтр для зольности

Оставляю спрос, серу и лимиты прежними. Ищу минимальное повышение порога зольности и при нём пересчитываю стоимость закупки.

In [8]:
def run_star(data):
    price, capacity, ash, sulfur = inputs(data)
    capacity = deficit_capacity(capacity)
    demand = data["demand"]
    ash_limit = data["stress_ash_limit_pct"]
    sulfur_limit = data["sulfur_limit_pct"]
    base = build_model(capacity, ash, sulfur, demand, ash_limit, sulfur_limit)
    model = dict(base, A_ub=np.column_stack((base["A_ub"], [-demand, 0])),
                 A_eq=np.column_stack((base["A_eq"], [0])),
                 bounds=[*map(tuple, base["bounds"]), (0, None)])
    elastic = np.r_[np.zeros(len(price)), 1]
    repair = solve_lp(elastic, label="minimum ash relaxation", **model)
    require_status(repair, 0)

    # В сертификате используем равный взвешенный коэффициент для S2 и S5.
    lam = (ash[4] - ash[1]) / (sulfur[1] - sulfur[4])
    weight = ash + lam * sulfur
    bound = float(weight @ fill_cheapest(weight, capacity, demand) / demand - lam * sulfur_limit)
    assert abs(repair.fun - (bound - ash_limit)) <= TOL

    repaired_limit = ash_limit + repair.fun
    restored = solve_lp(price, label="least cost at restored ash limit",
                        **build_model(capacity, ash, sulfur, demand, repaired_limit, sulfur_limit))
    require_status(restored, 0)
    checks = verify(restored.x, price, capacity, ash, sulfur, demand, repaired_limit,
                    sulfur_limit, restored.fun)
    checks.update(minimum_increase=float(repair.fun), certified_minimum_ash=bound,
                  repaired_ash_limit=float(repaired_limit),
                  achieved_ash_pct=float(ash @ restored.x / demand),
                  achieved_sulfur_pct=float(sulfur @ restored.x / demand))
    write_csv(STAR_RESULTS, "repair", ["original_ash_limit", "increase", "repaired_ash_limit", "cost"],
              [[ash_limit, repair.fun, repaired_limit, restored.fun]])
    write_csv(STAR_RESULTS, "solution", ["supplier", "tonnes"], zip(data["suppliers"], restored.x))
    write_check(STAR_RESULTS, "checks", checks)


run_with_log(run_star, data, STAR_RESULTS)

show_table(
    STAR_RESULTS / "repair.csv",
    {"original_ash_limit": "Исходный предел, %", "increase": "Поправка, п. п.",
     "repaired_ash_limit": "Новый предел, %", "cost": "Стоимость, руб."},
)

| Исходный предел, % | Поправка, п. п. | Новый предел, % | Стоимость, руб. |
| --- | --- | --- | --- |
| 7,9 | 0,609 | 8,509 | 5 943 833,571 |